#  ⭐ `dim_notificacoes`

In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings, fuzzy_merge##, normalize_duration, expandir_gravidade_wide
project_root = get_project_root() 

In [2]:
project_root

WindowsPath('C:/Users/janet/Documents/VigiMed/vigimed')

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/anvisa/Notificacoes/Notificacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['UF', 'TIPO_ENTRADA_VIGIMED', 'RECEBIDO_DE',
       'IDENTIFICACAO_NOTIFICACAO', 'DATA_INCLUSAO_SISTEMA',
       'DATA_ULTIMA_ATUALIZACAO', 'DATA_NOTIFICACAO', 'TIPO_NOTIFICACAO',
       'NOTIFICACAO_PARENT_CHILD', 'DATA_NASCIMENTO', 'IDADE_MOMENTO_REACAO',
       'GRUPO_IDADE', 'IDADE_GESTACIONAL_MOMENTO_REACAO', 'SEXO', 'GESTANTE',
       'LACTANTE', 'PESO_KG', 'ALTURA_CM', 'REACAO_EVENTO_ADVERSO_MEDDRA',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'RELACAO_MEDICAMENTO_EVENTO', 'NOME_MEDICAMENTO_WHODRUG',
       'ACAO_ADOTADA', 'NOTIFICADOR', 'ATUALIZADO'],
      dtype='object')

In [4]:
bronze.head(2)

,UF,TIPO_ENTRADA_VIGIMED,RECEBIDO_DE,IDENTIFICACAO_NOTIFICACAO,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,TIPO_NOTIFICACAO,NOTIFICACAO_PARENT_CHILD,DATA_NASCIMENTO,IDADE_MOMENTO_REACAO,GRUPO_IDADE,IDADE_GESTACIONAL_MOMENTO_REACAO,SEXO,GESTANTE,LACTANTE,PESO_KG,ALTURA_CM,REACAO_EVENTO_ADVERSO_MEDDRA,GRAVE,GRAVIDADE,DESFECHO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,ACAO_ADOTADA,NOTIFICADOR,ATUALIZADO
0,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300212656,20230928,20230928,None,Notificaçăo espontânea,None,19900131,30 ano,None,None,Feminino,Năo,Năo,68.0,165,Hemiparesia,Sim,Outro efeito clinicamente significativo,Recuperado,20210125,20210206,12 dia,Concomitante,Tamisa,Năo aplicável,Médico,2026-01-15
1,SP,Empresas Farmacęuticas,Empresa Farmacęutica,BR-ANVISA-300208322,20230901,20230901,None,Notificaçăo espontânea,None,None,None,None,None,Feminino,Năo,Năo,None,None,Cefaleia,Năo,Outro efeito clinicamente significativo,Desconhecido,20210122,None,None,Suspeito,CoronaVac,Năo aplicável,Consumidor ou outro năo profissional de saúde,2026-01-15


# 🥈 Silver

normalized data, medium quality

In [5]:
silver = bronze.copy()
silver.shape

(311134, 30)

## ✅ hash

In [6]:
from utils import build_row_hash

silver["HASH_BRONZE"] = build_row_hash(
    silver, 
    silver.columns.difference(['ATUALIZADO']).tolist(), 
    algo="sha256", 
    sep="|"
)

## ✅ Missing

In [7]:
silver.isna().sum()

UF                                   81507
TIPO_ENTRADA_VIGIMED                    70
RECEBIDO_DE                          55666
IDENTIFICACAO_NOTIFICACAO                0
DATA_INCLUSAO_SISTEMA                    0
DATA_ULTIMA_ATUALIZACAO                 70
DATA_NOTIFICACAO                    151881
TIPO_NOTIFICACAO                         0
NOTIFICACAO_PARENT_CHILD            310006
DATA_NASCIMENTO                      80710
IDADE_MOMENTO_REACAO                110538
GRUPO_IDADE                         183974
IDADE_GESTACIONAL_MOMENTO_REACAO    310802
SEXO                                 11687
GESTANTE                            160811
LACTANTE                            160814
PESO_KG                             189266
ALTURA_CM                           244428
REACAO_EVENTO_ADVERSO_MEDDRA           317
GRAVE                                69292
GRAVIDADE                           168932
DESFECHO                             49933
DATA_INICIO_HORA                     85453
DATA_FINAL_

In [8]:
hist_silver = silver.copy()
for col in silver.select_dtypes(include=['object']).columns:
    silver[col] = normalize_rows(silver[col])

In [9]:
hist_silver.isna().sum()

UF                                   81507
TIPO_ENTRADA_VIGIMED                    70
RECEBIDO_DE                          55666
IDENTIFICACAO_NOTIFICACAO                0
DATA_INCLUSAO_SISTEMA                    0
DATA_ULTIMA_ATUALIZACAO                 70
DATA_NOTIFICACAO                    151881
TIPO_NOTIFICACAO                         0
NOTIFICACAO_PARENT_CHILD            310006
DATA_NASCIMENTO                      80710
IDADE_MOMENTO_REACAO                110538
GRUPO_IDADE                         183974
IDADE_GESTACIONAL_MOMENTO_REACAO    310802
SEXO                                 11687
GESTANTE                            160811
LACTANTE                            160814
PESO_KG                             189266
ALTURA_CM                           244428
REACAO_EVENTO_ADVERSO_MEDDRA           317
GRAVE                                69292
GRAVIDADE                           168932
DESFECHO                             49933
DATA_INICIO_HORA                     85453
DATA_FINAL_

In [10]:
hist_silver.shape

(311134, 31)

## ✅ DATES

In [13]:
colunas_data = ["DATA_INCLUSAO_SISTEMA", "DATA_ULTIMA_ATUALIZACAO", "DATA_NOTIFICACAO", "DATA_NASCIMENTO", "DATA_INICIO_HORA", "DATA_FINAL_HORA"]

In [14]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311134 entries, 0 to 311133
Data columns (total 6 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   DATA_INCLUSAO_SISTEMA    311134 non-null  object
 1   DATA_ULTIMA_ATUALIZACAO  311064 non-null  object
 2   DATA_NOTIFICACAO         159253 non-null  object
 3   DATA_NASCIMENTO          230424 non-null  object
 4   DATA_INICIO_HORA         225681 non-null  object
 5   DATA_FINAL_HORA          157812 non-null  object
dtypes: object(6)
memory usage: 14.2+ MB


,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,DATA_NASCIMENTO,DATA_INICIO_HORA,DATA_FINAL_HORA
0,20230928,20230928,None,19900131,20210125,20210206
1,20230901,20230901,None,None,20210122,None
2,20231006,20231006,None,19710522,20210203,None
3,20250721,20250721,None,19741123,20210302,None
4,20230927,20230927,None,None,None,None


In [15]:
for col in colunas_data:
    if col in hist_silver.columns:
        hist_silver[col] = normalize_date_column(hist_silver[col])

In [16]:
hist_silver[colunas_data].info()
hist_silver[colunas_data].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311134 entries, 0 to 311133
Data columns (total 6 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   DATA_INCLUSAO_SISTEMA    311134 non-null  datetime64[ns]
 1   DATA_ULTIMA_ATUALIZACAO  311064 non-null  datetime64[ns]
 2   DATA_NOTIFICACAO         158960 non-null  datetime64[ns]
 3   DATA_NASCIMENTO          229298 non-null  datetime64[ns]
 4   DATA_INICIO_HORA         169043 non-null  datetime64[ns]
 5   DATA_FINAL_HORA          107847 non-null  datetime64[ns]
dtypes: datetime64[ns](6)
memory usage: 14.2 MB


,DATA_INCLUSAO_SISTEMA,DATA_ULTIMA_ATUALIZACAO,DATA_NOTIFICACAO,DATA_NASCIMENTO,DATA_INICIO_HORA,DATA_FINAL_HORA
0,2023-09-28,2023-09-28,NaT,1990-01-31,2021-01-25,2021-02-06
1,2023-09-01,2023-09-01,NaT,NaT,2021-01-22,NaT
2,2023-10-06,2023-10-06,NaT,1971-05-22,2021-02-03,NaT
3,2025-07-21,2025-07-21,NaT,1974-11-23,2021-03-02,NaT
4,2023-09-27,2023-09-27,NaT,NaT,NaT,NaT


In [17]:
hist_silver.shape

(311134, 31)

## ✅ Mappings

- TIPO_ENTRADA_VIGIMED
- RECEBIDO_DE
- IDENTIFICACAO_NOTIFICADOR
- TIPO_NOTIFICADOR
- NOTIFICACAO+PARENT_CHILD
- GRUPO_IDADE
- SEXO
- GESTANTE
- LACTANTE
- REACAO_ADVERSO_MEDDRA
- GRAVE
- GRAVIDADE
- DESFECHO
- RELACAO_MEDICAMENTO_EVENTO
- ACAO_ADOTADA
- NOTIFICADOR

In [ ]:
# usar referencia fat med new -> duracao